#### Download

In [ ]:
from manticore_data.loader import ManticoreDataLoader, ManticoreGeneration

ml = ManticoreDataLoader(ManticoreGeneration.Local)

In [ ]:
for i, vf in enumerate(ml.get_velocity_field(1024, 256)):
    vf.download(f'data/manticore/ml-{i}.hdf5')

## Use

In [ ]:
import numpy as np

from scipy.interpolate import RegularGridInterpolator
from astropy.cosmology import FlatLambdaCDM, Planck18
from astropy.coordinates import SkyCoord, CartesianDifferential, CartesianRepresentation
from astropy import units as u
from astropy.constants import c as speed_of_light



def correct_vpec(grids, v, ra, dec, zCMB, cosmo=Planck18, coordsys='icrs', max_iter=50):
    coords = SkyCoord(ra=ra, dec=dec, unit='deg').transform_to(coordsys)
    zPEC = 0

    zPECs = []
    for i in range(max_iter):
        zHD = (1+zCMB) / (1 + zPEC) - 1

        coords_3d = SkyCoord(coords, distance=cosmo.comoving_distance(zHD)).cartesian
        xyz = np.moveaxis(coords_3d.xyz.to('Mpc').value, 0, -1)

        v_x, v_y, v_z = RegularGridInterpolator(grids, v, bounds_error=False, fill_value=float('nan'))(xyz).T * u.km / u.s
        zPEC = (SkyCoord(coords_3d, v_x=v_x, v_y=v_y, v_z=v_z, representation_type=CartesianRepresentation, differential_type=CartesianDifferential).radial_velocity / speed_of_light).to('').value
        zPECs.append(zPEC)

    return (1+zCMB) / (1 + zPEC) - 1, np.stack(zPECs)

In [ ]:
from libech0ez.vpec import correct_cmb
import pandas as pd

ztf_name = 'ztfdr2-vlhq_nom'
ztf = correct_cmb(pd.read_csv(f'data/{ztf_name}.csv').set_index('ztfname').rename(columns={'z': 'zHEL'}))

In [ ]:
from scipy.stats import chi2

(Vext_abs_mean, Vext_abs_std), (Vext_l_mean, Vext_l_std), (Vext_b_mean, Vext_b_std), (sigmav_mean, sigmav_std) = (
    ((a+b)/2, abs(a-b)/2 / chi2(df=2).ppf(0.95)**0.5) for a, b in (
        (57, 379), (248, 342), (-44, 19), (77, 222)
    )
)
Vext_abs_mean, Vext_abs_std, sigmav_mean, sigmav_std

In [ ]:
import h5py
from random import randint, normalvariate
from tqdm import trange

COORDSYS = 'icrs'

zHDs = []
for _ in trange(1000):
    i = randint(0, 79)
    f = h5py.File(f'data/manticore/ml-{i}.hdf5')
    v = np.stack([f[f'p{i}'][:] / (f['density'][:] + 1e-10) for i in range(3)], -1)

    Vext = normalvariate(Vext_abs_mean, Vext_abs_std) * SkyCoord(
        l=normalvariate(Vext_l_mean, Vext_l_std) * u.deg,
        b=normalvariate(Vext_b_mean, Vext_b_std) * u.deg,
        frame='galactic'
    ).transform_to(COORDSYS).cartesian.xyz

    sigmav = normalvariate(sigmav_mean, sigmav_std)

    zHEL = np.clip(ztf['zHEL'] + np.random.randn(len(ztf)) * (
        ztf['zHEL_err']**2 +
        ((1+ ztf['zSUN']) * sigmav/3e5)**2
    )**0.5, 1e-3, float('inf'))

    zHDs.append(correct_vpec(
        [np.linspace(-500, 500, s, endpoint=False) for s in v.shape[:3]], v + Vext,
        ztf['ra_host'], ztf['dec_host'],
        zCMB=(1 + zHEL) / (1 + ztf['zSUN']) - 1,
        cosmo=FlatLambdaCDM(Om0=0.306, H0=68.1), coordsys=COORDSYS
    )[0])
zHDs = np.stack(zHDs)

In [ ]:
np.savez(f'data/{ztf_name}-manticore.npz', zHD=zHDs.mean(0), z_cov=np.cov(zHDs.T))